# Colab smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lstival/ssl_tutorial_sibgrapi2026/blob/main/notebooks/colab_smoke_test.ipynb)

Run this **before the tutorial session** to confirm the whole thing works on a real Colab
GPU runtime: fresh clone, real GPU, weights downloaded from the GitHub Release exactly the
way a participant's browser would get them -- not a local/CI simulation.

`tools/run_notebooks_ci.py` (used by the CPU-only GitHub Actions workflow on every push) runs
the same notebooks headless, but on CPU, so it cannot catch GPU-only failures (device/dtype
mismatches, Colab's GPU memory ceiling, a checkpoint saved with a CUDA-only tensor type,
etc). This notebook closes that gap. It reuses the same runner, just pointed at a GPU kernel.

**Runtime > Change runtime type > GPU**, then Runtime > Run all. Read the last cell's report --
it lists exactly which notebook (if any) broke, and why.

In [ ]:
import torch, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU on this runtime. Runtime > Change runtime type > GPU, then Runtime > Run all."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Fresh clone -- do NOT reuse a cached checkout, the point is to reproduce exactly what a
# participant's first run looks like.
import os

REPO_URL = "https://github.com/lstival/ssl_tutorial_sibgrapi2026.git"
REPO_DIR = "/content/ssl_tutorial_sibgrapi2026"

if os.path.isdir(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Colab already ships torch/torchvision/numpy/pandas/matplotlib/scikit-learn/tqdm/requests
# (see requirements.txt). Only the packages this smoke test itself needs (papermill, to run
# the other notebooks headlessly) plus the two RS-only extras are installed here.
!pip install -q papermill timm rasterio seaborn

In [ ]:
# Confirm the GitHub Release actually serves the real tensors before spending GPU time on
# notebooks that would only fail later trying to load them.
!python tools/verify_assets.py --remote

In [ ]:
# Run every teaching notebook end-to-end, on this GPU, downloading weights on demand exactly
# as a participant's notebook would. Takes longer than the CPU CI run; that's expected --
# this is the one that has to match Colab reality, not be fast.
!python tools/run_notebooks_ci.py --timeout 1800

## Reading the result

- **All PASS** -- the tutorial will run for participants as-is on Colab's free GPU tier.
- **Any FAIL** -- the cell above prints which notebook and the tail of the actual traceback
  (dependency, checkpoint download, or a GPU-only bug). Fix it and re-run this notebook from
  the top (fresh clone) before the session, don't just re-run the failed notebook in isolation
  -- the point is reproducing a participant's first run.